# Resident Risk Level Drivers — Explanatory Model

---

## 1. Problem Framing

### Business Problem

Every resident at Hearth Haven is assessed at intake with an initial risk level
(Low, Medium, High, Critical). Over time, that risk level changes based on their
experience in care. Understanding *what drives* a resident's current risk level
is essential for case planning: if social workers know which factors are most
strongly associated with higher or lower risk, they can direct interventions
toward the dimensions that matter most.

This pipeline answers the question: **Which resident characteristics and care
history factors are most strongly associated with a resident's current risk level,
and what is the estimated magnitude of each association?**

The deployed output is a **Key Risk Drivers panel** on the resident case dashboard,
showing the top factors associated with a resident's current risk assessment and
their direction — giving social workers a structured explanation of *why* a
resident is at their current risk level, not just what that level is.

### Who Cares About This

- **Social workers** — need to understand which dimensions of a case to focus
  intervention efforts on for residents with elevated risk.
- **Case supervisors** — need to identify systemic patterns across the caseload:
  which risk factors appear most frequently and are most modifiable.
- **Organization leadership** — needs to report to donors and partner agencies
  on what factors the organization's programs are addressing.

### Predictive vs. Explanatory

This pipeline uses an **explanatory approach**. The goal is to quantify which
factors are statistically associated with current risk level and by how much —
not to generate accurate predictions for individual residents.

The pipeline exploration confirmed this choice: all predictive CV scores produced
negative R² (worse than predicting the mean). With 61 residents and 60 features,
p >> n makes reliable prediction impossible at this sample size. The explanatory
approach is the right tool here — OLS with careful feature selection produces
defensible coefficient estimates even with small samples.

Coefficients are the primary output. They tell us: holding all other factors
constant, a one-unit increase in [feature] is associated with a [coefficient]-unit
change in current risk level (where 0=Low, 1=Medium, 2=High, 3=Critical).

### Success Metrics

- **Primary:** Coefficient significance (p-values), direction, and magnitude
- **Secondary:** Adjusted R² — model-level explanatory fit
- **Not primary:** Predictive R² on held-out data — the exploration confirmed
  prediction is not reliable at this sample size

### A Note on the Risk Scale

`current_risk_num` treats the risk ordinal (Low=0, Medium=1, High=2, Critical=3)
as a continuous numeric outcome for OLS purposes. This is a modeling simplification
— the true scale is ordinal, not continuous. Coefficients should be interpreted
as associations with movement along the risk spectrum, not as precise unit changes.
An ordinal regression would be more technically correct; OLS is used here for
interpretability and consistency with the course framework.

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

sys.path.append(os.path.dirname(os.path.abspath('.')))
os.chdir('..')

from functions.fn_domain_prep import prepare_residents
from functions.fn_prepare import define_features, split_data
from functions.fn_model_causal import (
    fit_causal_regression,
    get_coefficients,
    check_assumptions,
    check_vif,
    refit_with_robust_se,
    run_greedy_backward,
)

print("All imports successful.")

### 2.1 Load and Prepare Data

`prepare_residents()` encodes every cleaning and feature engineering decision from
`eda_residents.ipynb`. It tries Azure SQL first (via `appsettings.Development.json`)
and falls back to local CSVs automatically.

**Tables joined:** `residents`, `health_wellbeing_records`, `education_records`,
`process_recordings`, `incident_reports`, `home_visitations`, `intervention_plans`

**Key preparation decisions encoded:**
- Structural columns dropped: IDs, free text, zero-variance fields (`sex`)
- Dates parsed; `age_at_admission_days`, `days_in_care`, `length_of_stay_days` engineered
- Intentional nulls filled with domain-appropriate sentinels
- Six supporting tables aggregated to one row per resident via left joins
- `current_risk_num` engineered from `current_risk_level` via ordinal mapping
  (Low=0, Medium=1, High=2, Critical=3)

In [ ]:
df, NUMERIC, CATEGORICAL, DROP = prepare_residents()

TARGET = 'current_risk_num'

print(f"Dataset shape: {df.shape}")
print(f"Target mean: {df[TARGET].mean():.2f}  |  Std: {df[TARGET].std():.2f}")
print(f"\nTarget distribution:")
print(df[TARGET].value_counts().sort_index())

### 2.2 Feature Definition

`define_features()` is called with `DROP['current_risk_num']` — the per-target
drop list defined during EDA. It prevents:

- **Direct leakage:** `current_risk_level` is the string version of the target —
  including it makes the model trivially perfect
- **Derived leakage:** `initial_risk_num`, `initial_risk_level` — these are the
  *intake* snapshot and are leaky here since the model would be explaining the
  gap between intake and current, not explaining current risk independently
- **Cross-target contamination:** `reintegration_achieved`, `reintegration_status`,
  `progress_percent_latest`, `risk_improved`, `risk_escalated` excluded

In [ ]:
X, y = define_features(
    df,
    target=TARGET,
    numeric=NUMERIC,
    categorical=CATEGORICAL,
    drop_cols=DROP[TARGET],
)

categorical_in_X = [c for c in CATEGORICAL if c in X.columns]
numeric_in_X     = [c for c in NUMERIC     if c in X.columns]
X[categorical_in_X] = X[categorical_in_X].astype(str).replace({'nan': np.nan, '<NA>': np.nan})

print(f"Feature matrix: {X.shape[0]} rows × {X.shape[1]} features")
print(f"  Numeric:     {len(numeric_in_X)}")
print(f"  Categorical: {len(categorical_in_X)}")

### 2.3 Exploratory Confirmation

EDA was conducted in `eda_residents.ipynb`. The cells below confirm expected
associations between features and risk level before fitting the model.

In [ ]:
# Top numeric features by absolute correlation with current_risk_num
corr = X[numeric_in_X].corrwith(y).sort_values(key=abs, ascending=False)
print(f"Top 10 numeric features by |correlation| with {TARGET}:")
print(corr.head(10).round(3).to_string())

In [ ]:
# Mean risk level by key categorical features
for col in ['case_status', 'case_category', 'referral_source']:
    if col in X.columns:
        rate = (pd.concat([X[col], y], axis=1)
                  .groupby(col)[TARGET]
                  .agg(['mean', 'count'])
                  .rename(columns={'mean': 'mean_risk', 'count': 'n'})
                  .sort_values('mean_risk', ascending=False))
        print(f"Mean risk by {col}:")
        print(rate.round(3).to_string(), "\n")

In [ ]:
# Distribution of top correlated features by risk level
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
features_to_plot = ['incident_count', 'high_severity_count', 'safety_concern_rate']

for ax, feat in zip(axes, features_to_plot):
    if feat not in X.columns:
        continue
    for risk_val, label in {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Critical'}.items():
        subset = X.loc[y == risk_val, feat]
        if len(subset) > 0:
            ax.hist(subset, alpha=0.5, label=label, bins=15)
    ax.set_title(feat)
    ax.set_xlabel('Value')
    ax.legend(fontsize=7)

plt.suptitle('Key Feature Distributions by Current Risk Level', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Causal Model Specification

### 3.1 Train/Test Split

Even for explanatory models we hold out a test set. We do not use it to select
features — that would be leakage — but we report test performance in Section 4
as an honest measure of how well the explanatory model generalizes.

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, stratify=False)

### 3.2 Multicollinearity Check (VIF)

Variance Inflation Factor (VIF) measures how much each feature's variance is
inflated by correlation with other features. High VIF (> 10) makes coefficients
unstable and hard to interpret — two highly collinear features will have inflated
standard errors and unreliable p-values.

This step is specific to the explanatory pipeline. In predictive pipelines,
multicollinearity is handled implicitly by regularization.

In [ ]:
# Encode for statsmodels — needs a plain numeric matrix
X_train_enc = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_train_enc = X_train_enc.apply(pd.to_numeric, errors='coerce').fillna(0)

print(f"Encoded matrix: {X_train_enc.shape[0]} rows × {X_train_enc.shape[1]} columns")
print("Running VIF check — this may take a moment...")

In [ ]:
vif_df   = check_vif(X_train_enc, threshold=10.0)
high_vif = vif_df[vif_df['VIF'] > 10]['feature'].tolist()            if 'VIF' in vif_df.columns else []

print(f"Features with VIF > 10 ({len(high_vif)} found):")
print(high_vif if high_vif else "None — all features acceptable")

X_clean = X_train_enc.drop(columns=high_vif, errors='ignore')
print(f"\nMatrix after VIF cleanup: {X_clean.shape[1]} features remaining")

### 3.3 Initial OLS Fit

We fit an initial OLS model on the VIF-cleaned feature matrix to assess overall
model fit and identify which features are significant before running feature
selection.

In [ ]:
results = fit_causal_regression(X_clean, y_train)
print(results.summary())

### 3.4 OLS Assumption Checks

Five assumptions must hold for OLS coefficients to be unbiased and interpretable:
1. **Linearity** — the relationship between X and y is linear
2. **Independence** — residuals are not autocorrelated
3. **Normality** — residuals are approximately normally distributed
4. **Homoscedasticity** — residual variance is constant across predicted values
5. **No multicollinearity** — already checked with VIF above

With n=48 training rows and an ordinal dependent variable treated as continuous,
some assumption violations are expected. We check each and apply robust standard
errors if needed.

In [ ]:
verdicts = check_assumptions(results)

if verdicts.get('homoscedasticity', {}).get('verdict') != 'PASS':
    print("\n[ACTION] Homoscedasticity failed — refitting with HC3 robust standard errors")
    results = refit_with_robust_se(results)
    print(results.summary())

### 3.5 Greedy Backward Feature Selection

With 48 training rows, the initial model is overfit — too many features relative
to observations. Greedy backward selection iteratively removes the feature whose
removal most reduces validation RMSE, producing the most parsimonious model
where every remaining feature has earned its place.

This is preferred for causal regression over p-value stepwise selection because
it optimizes for predictive validity on the validation set, not just in-sample
significance — producing coefficient estimates that are more stable.

In [ ]:
from sklearn.model_selection import train_test_split as sk_split

# Carve a validation set from training data for greedy backward
# The test set stays locked and is not used here
X_tr, X_val, y_tr, y_val = sk_split(
    X_clean, y_train, test_size=0.25, random_state=42
)

print(f"Greedy backward: {X_clean.shape[1]} features → finding optimal subset")
print(f"Train: {len(X_tr)} rows  |  Val: {len(X_val)} rows")
print("Running — this may take a few minutes...")

trace, optimal_features = run_greedy_backward(
    X_tr, y_tr, X_val, y_val,
    numeric_features=numeric_in_X,
    categorical_features=categorical_in_X,
)

In [ ]:
# Plot the removal trace
plt.figure(figsize=(10, 4))
plt.plot(trace['n_features'], trace['val_rmse'], marker='o', color='steelblue', ms=4)
plt.axvline(trace.loc[trace['val_rmse'].idxmin(), 'n_features'],
            color='red', linestyle='--', label='Optimal')
plt.xlabel('Number of Features')
plt.ylabel('Validation RMSE')
plt.title('Greedy Backward Feature Selection — Validation RMSE')
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nOptimal feature set ({len(optimal_features)} features):")
print(optimal_features)

In [ ]:
# Refit on the optimal feature subset using the full training set
X_final = X_clean[optimal_features]
results_final = fit_causal_regression(X_final, y_train)

# Apply robust SE if homoscedasticity check failed above
# results_final = refit_with_robust_se(results_final)

print(results_final.summary())

---
## 4. Evaluation & Interpretation

### 4.1 Coefficient Table

The primary output of this explanatory pipeline — which features are
significantly associated with current risk level and in which direction.

In [ ]:
coef_df = get_coefficients(results_final, model_type='linear')

print("All coefficients (sorted by |coefficient|):")
print(coef_df[['feature', 'coefficient', 'std_err', 'p_value', 'significant']]
      .to_string(index=False))

print("\nSignificant features only (p < 0.05):")
sig = coef_df[coef_df['p_value'] < 0.05].sort_values('coefficient', ascending=False)
print(sig[['feature', 'coefficient', 'std_err', 'p_value', 'significant']]
      .to_string(index=False))

In [ ]:
# Visualize significant coefficients
if len(sig) > 0:
    sig_plot = sig.set_index('feature')['coefficient'].sort_values()
    colors   = ['coral' if v < 0 else 'steelblue' for v in sig_plot]
    sig_plot.plot(kind='barh',
                  figsize=(10, max(4, len(sig_plot) * 0.45)),
                  color=colors)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title('Significant Coefficients — Current Risk Level\n'
              '(Positive = associated with higher risk)', fontsize=12)
    plt.xlabel('Coefficient (risk units, where 1 unit = one risk tier)')
    plt.tight_layout()
    plt.show()
else:
    print("No significant features at p < 0.05. Review VIF cleanup and feature selection.")

### 4.2 Model Fit

In [ ]:
print("Model fit statistics:")
print(f"  R²:           {results_final.rsquared:.4f}")
print(f"  Adjusted R²:  {results_final.rsquared_adj:.4f}")
print(f"  F-statistic:  {results_final.fvalue:.4f}  (p = {results_final.f_pvalue:.6f})")
print(f"  Observations: {int(results_final.nobs)}")
print(f"  Features:     {len(results_final.params) - 1}")

# Honest comparison: how does the final model do on the test set?
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Encode test set with same columns as training
X_test_enc = pd.get_dummies(X_test, drop_first=True, dtype=int)
X_test_enc = X_test_enc.apply(pd.to_numeric, errors='coerce').fillna(0)
X_test_final = X_test_enc.reindex(columns=optimal_features, fill_value=0)

import statsmodels.api as sm
X_test_const = sm.add_constant(X_test_final, has_constant='add')
y_pred_test  = results_final.predict(X_test_const)

test_r2   = r2_score(y_test, y_pred_test)
test_rmse = float(np.sqrt(mean_squared_error(y_test, y_pred_test)))
baseline  = float(np.sqrt(mean_squared_error(y_test, np.full(len(y_test), y_train.mean()))))

print(f"\nTest set performance:")
print(f"  R²:            {test_r2:.4f}")
print(f"  RMSE:          {test_rmse:.4f}")
print(f"  Baseline RMSE: {baseline:.4f}  (predict-mean)")

### 4.3 Causal Interpretation

**How to read the coefficients:**

Each coefficient represents the estimated change in `current_risk_num` (on a
0=Low, 1=Medium, 2=High, 3=Critical scale) associated with a one-unit increase
in that feature, holding all other features constant.

For example, a coefficient of +0.4 on `high_severity_count` means that each
additional high-severity incident is associated with moving approximately 0.4
units up the risk scale — nearly half a risk tier — after controlling for other
factors.

**What significant features typically reveal:**

Features positively associated with higher risk tend to cluster into:

1. **Incident severity** — `high_severity_count`, `avg_severity`,
   `self_harm_incidents`, `runaway_attempts`. Residents with more severe incident
   histories show higher current risk levels. This is the most direct operational
   signal: incident records are both a symptom and a predictor of risk trajectory.

2. **Family and safety concerns** — `safety_concern_rate`,
   `family_cooperation_score`. Residents whose home visitation records show safety
   concerns and low family cooperation are associated with elevated risk. Family
   environment is a key modifiable factor in risk reduction.

3. **Case complexity** — `open_plans`, `total_plans`. Residents with more
   intervention plans (especially open/unachieved ones) tend to have higher risk
   levels, reflecting that more complex cases require more structured intervention.

Features negatively associated with risk (i.e., associated with lower risk):

4. **Care progress** — `plan_achievement_rate`, `progress_rate`. Residents who
   are completing their intervention goals and showing counseling progress are
   associated with lower current risk levels.

**What we cannot claim causally:**

- We cannot say that *reducing* incident counts will *cause* risk to decrease.
  Both incidents and risk level are driven by the same underlying circumstances —
  a resident in a genuinely dangerous situation will have both more incidents and
  higher risk.
- The model was estimated on a small sample (n=48 training rows). Coefficient
  estimates will shift as more residents are added. Treat direction as more
  reliable than magnitude at this sample size.
- OLS treats the ordinal risk scale as continuous. A one-unit change from
  Low→Medium is not necessarily the same magnitude as Medium→High in practice.

**Actionable insight (correlation-based, not causal):**

The association between incident severity metrics and current risk level provides
a structured basis for case review prioritization: residents with recent high-severity
incidents or elevated safety concern rates in visitation records warrant more
frequent case conferences, even if their formal risk classification has not yet
been updated by a social worker.

### 4.4 Limitations

- **Sample size:** n=61 residents. Coefficient estimates are directionally
  informative but have wide confidence intervals. The model should be retrained
  and reinterpreted quarterly as Hearth Haven accumulates its own case records.
- **Ordinal outcome treated as continuous:** OLS is an approximation here. If
  sample size grows sufficiently, an ordinal logistic regression would be more
  technically appropriate.
- **No temporal dimension:** All features are aggregated across the resident's
  full history. A resident with many incidents two years ago looks similar to
  one with recent incidents — a limitation the current feature set cannot resolve
  without time-windowed aggregation.

---
## 5. Causal and Relationship Analysis

The OLS model in Section 3 *is* the explanatory model for this pipeline — Section 5
in predictive notebooks fits a separate causal model, but here the causal analysis
is the primary output, not a supplement.

This section summarizes the key relationships found and contextualizes them against
what we would expect from child welfare research literature.

In [ ]:
# Summary: features by direction and significance
pos_sig = sig[sig['coefficient'] > 0][['feature', 'coefficient', 'p_value']]
neg_sig = sig[sig['coefficient'] < 0][['feature', 'coefficient', 'p_value']]

print("=== Features INCREASING risk (positive, significant) ===")
print(pos_sig.to_string(index=False) if len(pos_sig) > 0 else "None")

print("\n=== Features DECREASING risk (negative, significant) ===")
print(neg_sig.to_string(index=False) if len(neg_sig) > 0 else "None")

In [ ]:
# Correlation heatmap of significant numeric features vs target
sig_numeric = [f for f in sig['feature'].tolist() if f in X[numeric_in_X].columns]

if len(sig_numeric) >= 2:
    plot_cols = sig_numeric + [TARGET]
    corr_matrix = pd.concat([X[sig_numeric], y], axis=1).corr()

    fig, ax = plt.subplots(figsize=(max(6, len(sig_numeric)), max(5, len(sig_numeric))))
    im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr_matrix.columns)))
    ax.set_yticks(range(len(corr_matrix.columns)))
    ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(corr_matrix.columns, fontsize=8)
    plt.colorbar(im, ax=ax)
    plt.title('Correlation — Significant Features + Target')
    plt.tight_layout()
    plt.show()

---
## 6. Deployment

This is an explanatory pipeline — the primary deliverable is the coefficient table,
not a prediction endpoint. The deployment pattern is **Option A: static coefficient
report** served directly from Azure SQL by the .NET backend.

The coefficient table is saved to `models/` as a CSV. The .NET backend exposes it
via a GET endpoint. The React frontend renders the top significant features as a
"Key Risk Drivers" panel on the resident case page.

In [ ]:
os.makedirs('models', exist_ok=True)

# Save coefficient table
coef_path = 'models/current_risk_num_coefficients.csv'
coef_df.to_csv(coef_path, index=False)
print(f"Coefficient table saved: {coef_path}")

# Save significant features only (for the dashboard panel)
sig_path = 'models/current_risk_num_drivers.csv'
sig.to_csv(sig_path, index=False)
print(f"Significant drivers saved: {sig_path}")

# Print final summary for deployment reference
print(f"\n=== Key Risk Drivers (Significant at p < 0.05) ===")
print(sig[['feature', 'coefficient', 'p_value', 'significant']].to_string(index=False))

---
## 7. API Response Reference

This pipeline does not serve a live prediction endpoint. Instead, the .NET backend
exposes the precomputed coefficient table via a static GET endpoint.

```json
GET /api/analysis/risk-drivers

{
  "drivers": [
    {
      "feature": "string",
      "coefficient": "float",
      "direction": "positive | negative",
      "p_value": "float",
      "significant": "*** | ** | * | (ns)"
    }
  ],
  "model_version": "current_risk_num_v1",
  "generated_at": "ISO datetime"
}
```

**coefficient** — The estimated change in risk level (0–3 scale) associated with
a one-unit increase in that feature, holding all others constant. Positive values
are associated with higher risk; negative values with lower risk.

**direction** — Derived from the sign of the coefficient. Used by the React
frontend to color-code the driver panel (red = risk-increasing, green =
risk-decreasing).

**significant** — The p-value significance flag from the OLS model. Only features
with `p_value < 0.05` are surfaced in the dashboard panel.

---
### .NET Endpoint to add to `CaseController.cs`

```csharp
[HttpGet("risk-drivers")]
[Authorize]
public IActionResult GetRiskDrivers()
{
    // Serve the precomputed coefficient CSV as structured JSON
    // Regenerate by rerunning this notebook and redeploying models/current_risk_num_drivers.csv
    var driversPath = Path.Combine(_env.ContentRootPath, "models", "current_risk_num_drivers.csv");
    if (!System.IO.File.Exists(driversPath))
        return NotFound(new { error = "Risk drivers model not found" });

    var lines  = System.IO.File.ReadAllLines(driversPath);
    var header = lines[0].Split(',');
    var rows   = lines.Skip(1).Select(line => {
        var vals = line.Split(',');
        return new {
            feature     = vals[0],
            coefficient = double.Parse(vals[1]),
            direction   = double.Parse(vals[1]) > 0 ? "positive" : "negative",
            p_value     = double.Parse(vals[3]),
            significant = vals[4]
        };
    }).Where(r => r.p_value < 0.05).ToList();

    return Ok(new {
        drivers       = rows,
        model_version = "current_risk_num_v1",
        generated_at  = System.IO.File.GetLastWriteTimeUtc(driversPath).ToString("o")
    });
}
```

*No `endpoints.py` or `server.py` changes needed for this pipeline — the .NET
backend serves the static CSV directly.*

---
*Hearth Haven — IS 455 INTEX Pipeline*